# Cómputo Evolutivo — `Individuo` y `Poblacion`
Modelado basado en genética humana.

## Clase `Individuo`

Representa a una persona con sus genes.  
Cada gen es un rasgo (ej. `color_ojos = cafe`).  
La **aptitud** mide qué tan bien se adapta al entorno.

In [ ]:
import random

class Individuo:
    """
    Representa a una persona con su carga genética.

    Atributos
    ---------
    nombre     : str   — Nombre de la persona.
    genes      : dict  — Rasgos heredados {nombre_gen: alelo}.
                         Ej: {"color_ojos": "cafe", "estatura": "alta"}
    aptitud    : float — Qué tan bien se adapta al entorno (0.0 a 1.0).
    generacion : int   — Generación a la que pertenece.
    """

    def __init__(self, nombre: str, genes: dict, generacion: int = 0):
        self.nombre     = nombre
        self.genes      = genes   # genotipo
        self.aptitud    = 0.0
        self.generacion = generacion

    # ------------------------------------------------------------------
    # Fenotipo: los rasgos visibles (expresión del genotipo)
    # ------------------------------------------------------------------
    @property
    def fenotipo(self) -> dict:
        """Devuelve los rasgos visibles de la persona (sus genes expresados)."""
        return dict(self.genes)

    # ------------------------------------------------------------------
    # Aptitud
    # ------------------------------------------------------------------
    def calcular_aptitud(self, entorno: dict) -> float:
        """
        Calcula qué tan adaptado está el individuo al entorno.
        Cuenta cuántos genes coinciden con los rasgos favorecidos.

        entorno: dict {nombre_gen: alelo_favorecido}
                 Ej: {"resist_frio": "alta", "color_piel": "clara"}
        """
        coincidencias = sum(
            1 for gen, alelo in self.genes.items()
            if entorno.get(gen) == alelo
        )
        self.aptitud = coincidencias / len(self.genes)
        return self.aptitud

    # ------------------------------------------------------------------
    # Mutación: un gen cambia de valor aleatoriamente
    # ------------------------------------------------------------------
    def mutar(self, alelos_posibles: dict, probabilidad: float = 0.1) -> None:
        """
        Con probabilidad `probabilidad`, cada gen puede cambiar
        a un alelo distinto al actual.

        alelos_posibles: dict {nombre_gen: [lista de alelos]}
        """
        for gen in self.genes:
            if random.random() < probabilidad:
                opciones = [a for a in alelos_posibles[gen] if a != self.genes[gen]]
                if opciones:
                    self.genes[gen] = random.choice(opciones)

    # ------------------------------------------------------------------
    # Reproducción: mezcla de genes de dos padres
    # ------------------------------------------------------------------
    def reproducirse(self, otro: "Individuo", nombre_hijo: str) -> "Individuo":
        """
        Genera un hijo. Cada gen del hijo proviene
        aleatoriamente del padre o de la madre.
        """
        genes_hijo = {
            gen: random.choice([self.genes[gen], otro.genes[gen]])
            for gen in self.genes
        }
        return Individuo(
            nombre=nombre_hijo,
            genes=genes_hijo,
            generacion=max(self.generacion, otro.generacion) + 1
        )

    def __repr__(self) -> str:
        return (
            f"{self.nombre} (gen {self.generacion}) "
            f"aptitud={self.aptitud:.2f} | genes={self.genes}"
        )

---
## Clase `Poblacion`

Un grupo de personas que viven en el mismo entorno.  
Cada generación, los más aptos se reproducen y sus hijos heredan sus genes.

In [ ]:
class Poblacion:
    """
    Grupo de individuos que evolucionan juntos.

    Atributos
    ---------
    individuos       : list[Individuo] — Personas del grupo.
    entorno          : dict            — Rasgos favorecidos por el ambiente.
    alelos_posibles  : dict            — Variantes posibles de cada gen.
    generacion       : int             — Generación actual.
    """

    def __init__(self, individuos: list, entorno: dict, alelos_posibles: dict):
        self.individuos      = individuos
        self.entorno         = entorno
        self.alelos_posibles = alelos_posibles
        self.generacion      = 0

    # ------------------------------------------------------------------
    # Evaluación
    # ------------------------------------------------------------------
    def evaluar(self) -> None:
        """Calcula la aptitud de todos los individuos según el entorno."""
        for ind in self.individuos:
            ind.calcular_aptitud(self.entorno)

    def mejor(self) -> Individuo:
        """Retorna al individuo más adaptado."""
        return max(self.individuos, key=lambda i: i.aptitud)

    # ------------------------------------------------------------------
    # Evolución: una nueva generación
    # ------------------------------------------------------------------
    def evolucionar(self, probabilidad_mutacion: float = 0.1) -> None:
        """
        Genera la siguiente generación:
        1. Evalúa quién está mejor adaptado.
        2. Los dos más aptos se reproducen.
        3. Los hijos pueden mutar.
        4. Reemplaza la población.
        """
        self.evaluar()
        ordenados = sorted(self.individuos, key=lambda i: i.aptitud, reverse=True)

        hijos = []
        for i in range(0, len(ordenados) - 1, 2):
            padre = ordenados[i]
            madre = ordenados[i + 1]
            hijo  = padre.reproducirse(madre, f"h{self.generacion+1}_{i//2+1}")
            hijo.mutar(self.alelos_posibles, probabilidad_mutacion)
            hijos.append(hijo)

        self.individuos  = ordenados[:2] + hijos   # mejores padres + hijos
        self.generacion += 1

    # ------------------------------------------------------------------
    # Cambio de entorno (nueva presión evolutiva)
    # ------------------------------------------------------------------
    def cambiar_entorno(self, nuevo_entorno: dict) -> None:
        """Cambia el entorno — simula migración o cambio climático."""
        self.entorno = nuevo_entorno

    def __repr__(self) -> str:
        apt_media = sum(i.aptitud for i in self.individuos) / len(self.individuos)
        return (
            f"Generacion {self.generacion} | "
            f"{len(self.individuos)} individuos | "
            f"aptitud media: {apt_media:.2f}"
        )

---
## Demostración

Una pequeña tribu migra al norte. El entorno frío favorece piel clara y alta resistencia al frío.

In [ ]:
random.seed(42)

ALELOS = {
    "color_piel" : ["oscura", "media", "clara"],
    "resist_frio": ["baja",   "media", "alta"],
    "tipo_sangre": ["A", "B", "O", "AB"],
}

ENTORNO_NORTE = {"color_piel": "clara", "resist_frio": "alta", "tipo_sangre": "O"}

poblacion = Poblacion(
    individuos=[
        Individuo("Ata",  {"color_piel": "oscura", "resist_frio": "baja",  "tipo_sangre": "O"}),
        Individuo("Biru", {"color_piel": "media",  "resist_frio": "media", "tipo_sangre": "A"}),
        Individuo("Cora", {"color_piel": "clara",  "resist_frio": "alta",  "tipo_sangre": "B"}),
        Individuo("Danu", {"color_piel": "media",  "resist_frio": "alta",  "tipo_sangre": "O"}),
    ],
    entorno=ENTORNO_NORTE,
    alelos_posibles=ALELOS,
)

for _ in range(4):
    poblacion.evolucionar(probabilidad_mutacion=0.2)
    print(poblacion)

print()
print("Individuo más adaptado:", poblacion.mejor())